# Конспект. Модуль 7: Гибридные системы

**Курс:** Мини-курс RecSys (13 модулей)
**Модуль:** 7 из 13 — «Гибридные системы»
**Цель модуля:** формализовать то, к чему мы уже фактически пришли в конце Модуля 6 — вместо выбора **одного** алгоритма «или-или», научиться системно комбинировать несколько источников сигнала в единое решение. В разделе 6.4.3 мы увидели, что шесть разных методов согласованно предсказали высокую оценку U1 для I4 — этот модуль даёт формальный инструментарий для того, чтобы **использовать** такое согласие (и корректно обрабатывать несогласие) на практике, а не просто constatировать его.

**Связь с предыдущими модулями:** каждый одиночный алгоритм, разобранный в Модулях 3–6, имеет структурную слабость, зеркально закрываемую другим алгоритмом — это не совпадение, а следствие того, что разные методы используют принципиально разные источники информации (сравните 3.3–4.2 с 6.1.3–6.1.4). Гибридизация — это не «косметическое улучшение процента точности», а прямой инженерный ответ на этот факт.

## 7.1 Зачем нужны гибриды

### 7.1.1 Систематизация проблем и решений

| Проблема | Модуль, где обнаружена | Решение через гибридизацию |
|:---|:---:|:---|
| Холодный старт **нового товара** — коллаборативные методы (3–5) не работают без единой оценки | 1.5.1, 5.1 | Контентные фичи (Модуль 6) работают с первого дня — не нужна история взаимодействий |
| Холодный старт **нового пользователя** — контентная модель может использовать разве что демографию, но не полноценный вкус | 1.5.1, 6.1.3 | Популярное / коллаборативное — как fallback после накопления первых нескольких взаимодействий |
| Низкая **serendipity** контентной модели — рекомендует «ещё то же самое» | 6.1.4 | Коллаборативная фильтрация добавляет неожиданность за счёт паттернов совместного потребления, недоступных чистому анализу контента |
| Popularity Bias чистой CF — модель тяготеет к уже популярному | 1.5.3 | Контентные фичи позволяют находить нишевые, но подходящие конкретному пользователю товары независимо от их общей популярности |

### 7.1.2 Главный принцип — независимые источники ошибок компенсируют друг друга

Ключевая идея, которую стоит унести с этого раздела, — не «гибрид лучше, потому что использует больше данных», а точнее: **если два метода совершают ошибки по разным, не связанным друг с другом причинам, их комбинация статистически надёжнее любого из них по отдельности**. Коллаборативная фильтрация ошибается там, где мало данных о взаимодействиях (разреженность, холодный старт). Контентная фильтрация ошибается там, где признаки товара не отражают реальную причину, по которой он нравится (неявные, «вкусовые» закономерности). Это **разные** источники ошибок — именно поэтому их комбинация работает, а не просто «усредняет два одинаково несовершенных мнения».

## 7.2 Типы гибридных архитектур

### 7.2.1 Weighted (взвешенное объединение)

**Формула:**

In [ ]:
score_hybrid(u, i) = α × score_collab(u, i) + (1-α) × score_content(u, i)

**Критически важный практический нюанс, который часто упускают:** методы, которые мы комбинируем, обычно возвращают оценки **в разных шкалах**. Матричная факторизация (Модуль 5) предсказывает рейтинг в исходной шкале (например, 1–5). Контентная фильтрация (Модуль 6) возвращает косинусное сходство (0–1). **Прежде чем взвешивать, оценки необходимо привести к общей шкале** — иначе параметр `α` не будет иметь чёткого содержательного смысла (небольшое изменение `α` может непропорционально сильно "перетягивать" результат в сторону той шкалы, где абсолютные числа больше).

**Полный проверенный численный пример** — комбинируем предсказание Funk SVD (Модуль 5.4.4) с контентным сходством (Модуль 6.4.2) для той же пары U1/I4 и U1/I5. Content-based сходство сначала линейно приводится к шкале 1–5 (`1 + 4×cosine`):

In [ ]:
Funk SVD:                    I4 = 4.939,  I5 = 4.323
Content-based (шкала 1-5):   I4 = 3.619,  I5 = 1.571

**Динамика гибридной оценки при разных `α` (реально вычислено):**

| α | hybrid(I4) | hybrid(I5) | Разрыв (I4 − I5) |
|:---:|:---:|:---:|:---:|
| 1.00 (чистый CF) | 4.939 | 4.323 | 0.616 |
| 0.75 | 4.609 | 3.635 | 0.974 |
| 0.50 | 4.279 | 2.947 | 1.332 |
| 0.25 | 3.949 | 2.259 | 1.690 |
| 0.00 (чистый content) | 3.619 | 1.571 | 2.048 |

**Наблюдение:** в этом конкретном примере оба метода **согласны** в направлении (I4 предпочтительнее I5) — поэтому ранжирование не меняется ни при каком `α`, меняется только «уверенность» (разрыв между оценками). Это важно понимать честно: гибридизация не всегда меняет итоговое решение — иногда она просто увеличивает или уменьшает нашу уверенность в уже согласованном выводе. По-настоящему `α` становится решающим фактором, когда методы **расходятся** в ранжировании — разберём это на гипотетическом примере.

**Гипотетический пример смены победителя (иллюстративный, не из основных данных курса):** пусть у товара I4 сильный коллаборативный сигнал, но слабый контентный (`collab=4.939, content=3.619`, как и раньше), а у гипотетического товара I6 — наоборот, слабый коллаборативный, но сильный контентный сигнал (`collab=3.5, content=4.9`, — например, товар совсем новый, взаимодействий мало, но по метаданным прекрасно подходит пользователю):

| α | hybrid(I4) | hybrid(I6) | Победитель |
|:---:|:---:|:---:|:---|
| 0.000 | 3.619 | 4.900 | I6 |
| 0.200 | 3.883 | 4.620 | I6 |
| **0.471** | **4.241** | **4.241** | **точка равновесия** |
| 0.700 | 4.543 | 3.920 | I4 |
| 1.000 | 4.939 | 3.500 | I4 |

**Это и есть содержательный смысл подбора `α` через кросс-валидацию (раздел 7.3):** найти ту точку баланса между двумя источниками сигнала, которая на реальных, ранее не виденных данных даёт наилучшее качество рекомендаций — а не интуитивно выбрать «50 на 50».

### 7.2.2 Switching (переключение)

**Идея:** не смешивать оценки постоянно, а **выбирать**, какой алгоритм использовать целиком, по некоторому правилу (чаще всего — по объёму накопленных данных о пользователе).

**Пример правила:** `если число оценок пользователя < порога -> используем контентную модель; иначе -> коллаборативную`.

**Проверка правила на нашем сквозном примере:** у U1 всего 3 оценки (`I1, I2, I3`). Если задать достаточно строгий порог (например, `threshold=5`, что вполне разумно для реальной системы, где статистическая надёжность коллаборативного сигнала начинает появляться позже, чем 3 взаимодействия), **U1 в этой архитектуре был бы направлен именно на контентную рекомендацию** (Модуль 6.4.2, `profile_U1`) — несмотря на то, что в этом курсе мы всё равно успешно применили к нему и UB-CF, и IB-CF, и матричную факторизацию для учебных целей. Это показывает, что выбор порога — не формальность, а содержательное архитектурное решение: слишком низкий порог допускает ненадёжную статистику коллаборативной фильтрации (Модуль 3.3.4 — проблема значимости сходства при малом пересечении), слишком высокий — слишком долго держит пользователей на менее персонализированной контентной модели.

### 7.2.3 Feature Combination (комбинация на уровне признаков)

**Идея:** не комбинировать **выходы** двух отдельных моделей (как в 7.2.1), а добавить контентные признаки **внутрь** самой модели матричной факторизации — то есть контентная информация используется на входе, а не на выходе.

**Практическая реализация:** вместо того чтобы обучать вектор `q_i` полностью «с нуля» из истории взаимодействий (Модуль 5.4), можно инициализировать его TF-IDF-вектором товара (Модуль 6.3) или добавить регуляризационный член, «притягивающий» `q_i` к контентному вектору:

In [ ]:
L = Σ(r_ui - p_u·q_i)² + λ₁(||p_u||² + ||q_i||²) + λ₂||q_i - content_vector_i||²

Для нового товара, у которого ещё нет обучающих данных для первого слагаемого, второе слагаемое (`λ₂`) всё равно «подтягивает» `q_i` к разумному, контентно-обоснованному значению — прямое, элегантное решение проблемы cold start товара **внутри** одной модели, а не через отдельный fallback-механизм, как в Switching.

### 7.2.4 Meta-level (мета-уровень)

**Идея:** результат одной модели (например, эмбеддинги пользователей/товаров, обученные ALS в Модуле 5.5) становится **входным признаком** для совершенно отдельной, второй модели — точно так же, как вы уже используете инженерные признаки (Feature Store) в качестве входа для LightGBM в проекте FraudGuard.

**Прямая связь с будущими модулями курса:** это ровно то, что будет происходить в production-архитектуре Модуля 11 — скор из ALS (`p_u · q_i`) станет всего лишь **одним из многих признаков**, наравне с популярностью, контентным сходством и контекстными признаками, поданных в LightGBM-модель ранжирования (Модуль 9, 11.4). Meta-level гибридизация — это, по сути, первое появление в курсе идеи «модель как генератор признаков для другой модели», которая станет центральной темой оставшейся части курса.

### 7.2.5 Cascade (каскадная гибридизация)

**Идея:** модели применяются **последовательно**, а не параллельно — первая модель (обычно быстрая и грубая) отбирает ограниченный список кандидатов, вторая (медленнее, но точнее) их ранжирует.

**Пример:** коллаборативная модель (Модуль 4 или 5) отбирает, скажем, 100 товаров-кандидатов из полного каталога, затем контентная модель более тщательно ранжирует именно эти 100 — а не весь каталог целиком.

**Это прямой прообраз архитектуры Модуля 11** (Retrieval + Ranking + Re-ranking) — на самом деле вся идея двухстадийных production-систем, к которой курс придёт в третьей четверти материала, **уже** сформулирована здесь, просто в терминах двух конкретных алгоритмов (коллаборативного и контентного), а не абстрактных «быстрой» и «точной» стадий. Стоит явно это отметить: если Cascade понятен на уровне этого раздела, Модуль 11 будет восприниматься не как новая идея, а как её масштабирование на промышленные объёмы данных.

### 7.2.6 Сводная таблица пяти архитектур

| Архитектура | Когда применяется параллельно/последовательно | Где решается cold start | Прообраз в будущих модулях |
|:---|:---|:---|:---|
| Weighted | Параллельно, оценки смешиваются | Частично (через вес α) | — |
| Switching | Выбор одной модели целиком | Полностью (отдельная ветка для холодных) | — |
| Feature Combination | Единая модель с составным входом | Полностью (регуляризация к контенту) | Two-Tower модели (Модуль 10) |
| Meta-level | Выход одной модели -> вход другой | Косвенно | Ranking stage (Модуль 9, 11) |
| Cascade | Строго последовательно | Косвенно | Retrieval + Ranking (Модуль 11) |

## 7.3 Практика

### 7.3.1 Weighted Hybrid с подбором α через кросс-валидацию

In [ ]:
import numpy as np
from sklearn.model_selection import KFold

def normalize_to_rating_scale(scores: np.ndarray, r_min=1, r_max=5) -> np.ndarray:
    """Приведение произвольной шкалы (например, косинуса 0-1) к шкале рейтингов."""
    s_min, s_max = scores.min(), scores.max()
    if s_max == s_min:
        return np.full_like(scores, (r_min+r_max)/2)
    return r_min + (r_max - r_min) * (scores - s_min) / (s_max - s_min)

def weighted_hybrid_score(collab_scores: np.ndarray, content_scores: np.ndarray, alpha: float) -> np.ndarray:
    content_scaled = normalize_to_rating_scale(content_scores)
    return alpha * collab_scores + (1 - alpha) * content_scaled

def tune_alpha(collab_scores, content_scores, true_ratings, alphas=np.linspace(0, 1, 21)):
    """Подбор оптимального alpha по RMSE на отложенной выборке (temporal split, Модуль 1.7)."""
    results = {}
    for alpha in alphas:
        preds = weighted_hybrid_score(collab_scores, content_scores, alpha)
        rmse = np.sqrt(np.mean((preds - true_ratings) ** 2))
        results[alpha] = rmse
    best_alpha = min(results, key=results.get)
    return best_alpha, results

**Важное методологическое замечание для полномасштабного эксперимента на MovieLens:** в отличие от игрушечного примера 7.2.1 (где у нас нет истинных значений для I4/I5 — они были специально сконструированы как неизвестные, поэтому оценить RMSE честно нельзя), на реальном датасете нужно построить **temporal split** (Модуль 1.7), обучить обе базовые модели **только на train**, и подбирать `α` по RMSE на **отложенных, но реально известных** оценках test-части — иначе подбор параметра сам станет источником утечки данных.

**Задание:** построить график `RMSE(α)` для `α ∈ [0, 1]` с шагом 0.05 на MovieLens. Ожидаемая форма кривой — обычно она не монотонна: слишком большой вес чистой CF (`α->1`) страдает от разреженности на редких товарах, слишком большой вес чистого контента (`α->0`) теряет точность на товарах с богатой историей взаимодействий — оптимум обычно находится где-то посередине, но его точное положение зависит от плотности конкретного датасета.

### 7.3.2 Switching Hybrid

In [ ]:
def switching_hybrid_recommend(user_id, user_ratings_count: dict, threshold: int,
                                 content_recommender, collab_recommender, k=10):
    """
    user_ratings_count: {user_id: число_оценок} — предвычислено заранее.
    """
    if user_ratings_count.get(user_id, 0) < threshold:
        # "Холодный" пользователь — используем контентную модель (Модуль 6.5.1)
        return content_recommender.recommend(user_id, k=k), 'content-based'
    else:
        # "Тёплый" пользователь — используем коллаборативную модель
        return collab_recommender.recommend(user_id, k=k), 'collaborative'

# Проверка на нашем сквозном примере: у U1 всего 3 оценки
user_ratings_count = {'U1': 3, 'U2': 4, 'U3': 5, 'U4': 4, 'U5': 4}
recs, method_used = switching_hybrid_recommend(
    'U1', user_ratings_count, threshold=5,
    content_recommender=recommender,  # из Модуля 6.5.1
    collab_recommender=None,          # не используется в этой ветке
)
print(f"U1 направлен на: {method_used}")  # ожидаем 'content-based' при threshold=5

**Задание:** реализовать полный Switching Hybrid на MovieLens с `threshold ∈ {3, 5, 10, 20}`, и для каждого значения порога измерить долю пользователей, которые в реальном датасете попадают в «холодную» (контентную) ветку. Обсудить: как эта доля меняется с ростом порога, и какой практический компромисс это отражает (больше пользователей на менее персонализированной модели против более надёжной статистики для тех, кто остаётся на CF).

### 7.3.3 Вопросы для самопроверки

1. В разделе 7.2.1 подчёркивается, что перед взвешиванием оценки нужно привести к общей шкале. Что конкретно пошло бы не так, если бы мы усредняли `α × rating_1_to_5 + (1-α) × cosine_0_to_1` **без** предварительной нормализации, особенно при малых `α`?
2. Почему в примере со «сменой победителя» (7.2.1) точка равновесия оказалась именно при `α=0.471`, а не при интуитивно ожидаемых `α=0.5`? Проследите, как это связано с конкретными значениями `collab` и `content` для I4 и I6 — что случится с точкой равновесия, если увеличить `content_I6` (сделать контентный сигнал ещё сильнее)?
3. Объясните разницу между Feature Combination (7.2.3) и Meta-level (7.2.4) гибридизацией — обе используют «выход одной модели как вход другой», в чём принципиальное отличие?
4. Мы уже отметили, что Cascade (7.2.5) — прообраз архитектуры Модуля 11. Опишите своими словами, чем именно (помимо масштаба данных) production Retrieval+Ranking система будет отличаться от простого Cascade из двух алгоритмов этого курса.

## Глоссарий модуля 7

| Термин | Короткое определение |
|:---|:---|
| Weighted Hybrid | Взвешенная сумма оценок нескольких моделей |
| Switching Hybrid | Выбор одной из нескольких моделей целиком, по правилу |
| Feature Combination | Встраивание контентных признаков внутрь модели факторизации |
| Meta-level Hybrid | Использование выхода одной модели как входного признака другой |
| Cascade Hybrid | Последовательное применение: быстрая модель отбирает, точная — ранжирует |
| Нормализация шкал | Приведение оценок разных моделей к общему диапазону перед комбинированием |

**Связь со следующим модулем:** до сих пор мы оценивали качество моделей интуитивно («предсказание выглядит разумным», «направление совпадает») или простым RMSE. Модуль 8 формализует полный набор метрик, необходимых для честного, количественного сравнения — в том числе для по-настоящему обоснованного подбора `α` в Weighted Hybrid (7.3.1) и порога в Switching Hybrid (7.3.2), вместо интуитивных догадок.